In [ ]:
#!/usr/bin/env python3
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.5 - CLASIFICACIÓN (SOLO ENTRENAMIENTO)
# ===================================================================
# DESCRIPCIÓN: Entrena modelos de clasificación y guarda resultados.
#              Las gráficas se generan en un script separado.
# Clases:
#   0 — Empty  : 0 personas
#   1 — Low    : 1–20 personas
#   2 — Medium : 21–35 personas
#   3 — High   : >35 personas
# ===================================================================

import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURACIÓN
# ==========================================
INPUT_DIR  = 'ml_normalized_grouped'
OUTPUT_DIR = 'ml_results_classification_4clases'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CLASSROOM = '1.5'

# ==========================================
# 1. CARGA DE DATOS
# ==========================================
print("\n" + "="*70)
print(f"AULA {CLASSROOM} - CLASIFICACIÓN (ENTRENAMIENTO) - 4 CLASES")
print("="*70)
print("\n1. LOADING DATA...")

try:
    X_train_norm = pd.read_excel(os.path.join(INPUT_DIR, 'X_train_normalised.xlsx')).values
    X_test_norm  = pd.read_excel(os.path.join(INPUT_DIR, 'X_test_normalised.xlsx')).values
    X_train_raw  = pd.read_excel(os.path.join(INPUT_DIR, 'X_train.xlsx')).values
    X_test_raw   = pd.read_excel(os.path.join(INPUT_DIR, 'X_test.xlsx')).values
    y_train_reg  = pd.read_excel(os.path.join(INPUT_DIR, 'y_train.xlsx')).values.ravel()
    y_test_reg   = pd.read_excel(os.path.join(INPUT_DIR, 'y_test.xlsx')).values.ravel()
    print(f"   Train: {len(y_train_reg)}, Test: {len(y_test_reg)}")
except FileNotFoundError as e:
    print(f"   ERROR: {e}")
    exit(1)

# ==========================================
# 2. CONVERSIÓN A CATEGORÍAS (4 CATEGORÍAS)
# ==========================================
print("\n2. CONVERTING TO CATEGORIES...")

def to_category(y):
    # 0: Empty (0), 1: Low (1-20), 2: Medium (21-35), 3: High (>35)
    return np.select(
        [
            y == 0,
            (y >= 1)  & (y <= 20),
            (y >= 21) & (y <= 35),
            y > 35
        ],
        [0, 1, 2, 3]
    )

y_train = to_category(y_train_reg)
y_test  = to_category(y_test_reg)
class_names = ['Empty(0)', 'Low(1-20)', 'Medium(21-35)', 'High(36+)']

print("   Distribución de clases:")
for cat, name in enumerate(class_names):
    train_count = np.sum(y_train == cat)
    test_count  = np.sum(y_test == cat)
    print(f"      {name}: train={train_count} ({train_count/len(y_train):.1%}), test={test_count} ({test_count/len(y_test):.1%})")

# Advertencia si alguna clase tiene muy pocas muestras
min_class = min(np.sum(y_train == c) for c in range(4))
if min_class < 5:
    print(f"\n   ⚠️  AVISO: La clase más pequeña tiene solo {min_class} muestras en train.")

# ==========================================
# 3. MODELOS Y VALIDACIÓN
# ==========================================
print("\n3. DEFINING MODELS & VALIDATION...")

# Modelos que necesitan datos normalizados
needs_scaling = {
    'Logistic Regression', 'SVM', 'KNN (k=5)', 'KNN (k=10)',
    'Naive Bayes', 'MLP (Neural Network)'
}

models = {
    # --- Ensemble ---
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=200, max_depth=10,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5,
        learning_rate=0.05, random_state=42
    ),
    'AdaBoost': AdaBoostClassifier(
        n_estimators=200, learning_rate=0.5,
        random_state=42
    ),
    
    # --- Linear ---
    'Logistic Regression': LogisticRegression(
        max_iter=2000, class_weight='balanced',
        random_state=42
    ),
    
    # --- SVM ---
    'SVM': SVC(
        kernel='rbf', C=10, gamma='scale',
        class_weight='balanced', random_state=42
    ),
    
    # --- Distance-based ---
    'KNN (k=5)': KNeighborsClassifier(
        n_neighbors=5, weights='distance', n_jobs=-1
    ),
    'KNN (k=10)': KNeighborsClassifier(
        n_neighbors=10, weights='distance', n_jobs=-1
    ),
    
    # --- Tree ---
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5, class_weight='balanced',
        random_state=42
    ),
    
    # --- Probabilistic ---
    'Naive Bayes': GaussianNB(),
    
    # --- Neural Network ---
    'MLP (Neural Network)': MLPClassifier(
        hidden_layer_sizes=(100, 50), activation='relu',
        solver='adam', max_iter=1000, random_state=42,
        early_stopping=True, validation_fraction=0.1
    ),
}

# Ajustar n_splits según la clase más pequeña
min_class_count = min(np.sum(y_train == c) for c in range(4))
n_splits = min(5, min_class_count) if min_class_count >= 2 else 2
cv_splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

print(f"   Modelos: {len(models)} | CV: StratifiedKFold({n_splits})")
print(f"   Con normalización: {sorted([n for n in needs_scaling if n in models])}")
print(f"   Sin normalización: {sorted(set(models.keys()) - needs_scaling)}")

# ==========================================
# 4. ENTRENAMIENTO CON DIAGNÓSTICO
# ==========================================
print("\n4. TRAINING MODELS WITH DIAGNOSIS...")
results = {}
trained_models = {}
predictions = {}

for name, model in models.items():
    X_tr = X_train_norm if name in needs_scaling else X_train_raw
    X_te = X_test_norm if name in needs_scaling else X_test_raw
    data_type = 'normalised' if name in needs_scaling else 'raw'
    print(f"\n   ▶ {name} ({data_type})...")

    try:
        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)

        # Métricas generales
        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1w  = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        f1m  = f1_score(y_test, y_pred, average='macro', zero_division=0)
        
        # Métricas por clase
        f1_per_class = f1_score(y_test, y_pred, average=None, zero_division=0)
        rec_per_class = recall_score(y_test, y_pred, average=None, zero_division=0)
        
        f1_empty  = f1_per_class[0]
        f1_low    = f1_per_class[1]
        f1_medium = f1_per_class[2]
        f1_high   = f1_per_class[3]
        rec_high  = rec_per_class[3]
        rec_empty = rec_per_class[0]

        # Validación cruzada para diagnóstico
        train_accs, val_accs, train_f1s, val_f1s = [], [], [], []
        for train_idx, val_idx in cv_splitter.split(X_tr, y_train):
            X_tr_fold, y_tr_fold = X_tr[train_idx], y_train[train_idx]
            X_val_fold, y_val_fold = X_tr[val_idx], y_train[val_idx]
            fold_model = clone(model)
            fold_model.fit(X_tr_fold, y_tr_fold)
            y_pred_train_fold = fold_model.predict(X_tr_fold)
            y_pred_val_fold   = fold_model.predict(X_val_fold)
            train_accs.append(accuracy_score(y_tr_fold, y_pred_train_fold))
            val_accs.append(accuracy_score(y_val_fold, y_pred_val_fold))
            train_f1s.append(f1_score(y_tr_fold, y_pred_train_fold, average='weighted', zero_division=0))
            val_f1s.append(f1_score(y_val_fold, y_pred_val_fold, average='weighted', zero_division=0))

        train_acc_mean = np.mean(train_accs)
        val_acc_mean   = np.mean(val_accs)
        train_f1_mean  = np.mean(train_f1s)
        val_f1_mean    = np.mean(val_f1s)
        acc_gap = train_acc_mean - val_acc_mean
        f1_gap  = train_f1_mean - val_f1_mean

        cv_scores = cross_val_score(model, X_tr, y_train, cv=cv_splitter, scoring='f1_weighted')

        results[name] = {
            'Accuracy':      round(acc, 4),
            'Precision':     round(prec, 4),
            'Recall':        round(rec, 4),
            'F1_weighted':   round(f1w, 4),
            'F1_macro':      round(f1m, 4),
            'F1_Empty':      round(f1_empty, 4),
            'F1_Low':        round(f1_low, 4),
            'F1_Medium':     round(f1_medium, 4),
            'F1_High':       round(f1_high, 4),
            'Recall_High':   round(rec_high, 4),
            'Recall_Empty':  round(rec_empty, 4),
            'CV_F1_mean':    round(cv_scores.mean(), 4),
            'CV_F1_std':     round(cv_scores.std(), 4),
            'Train_Acc':     round(train_acc_mean, 4),
            'Val_Acc':       round(val_acc_mean, 4),
            'Acc_Gap':       round(acc_gap, 4),
            'Train_F1':      round(train_f1_mean, 4),
            'Val_F1':        round(val_f1_mean, 4),
            'F1_Gap':        round(f1_gap, 4),
        }
        trained_models[name] = model
        predictions[name]    = y_pred

        print(f"      Test Acc: {acc:.4f} | F1w: {f1w:.4f} | F1m: {f1m:.4f}")
        print(f"      Empty: F1={f1_empty:.4f} | Low: F1={f1_low:.4f} | Medium: F1={f1_medium:.4f} | High: F1={f1_high:.4f}")
        print(f"      Recall_High: {rec_high:.4f} | Recall_Empty: {rec_empty:.4f}")
        print(f"      CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        print(f"      Train Acc: {train_acc_mean:.4f} | Val Acc: {val_acc_mean:.4f} | Gap: {acc_gap:.4f}")
    
    except Exception as e:
        print(f"      ❌ ERROR: {e}")
        continue

# ==========================================
# 5. DIAGNÓSTICO Y RESUMEN
# ==========================================
print("\n" + "="*70)
print("5. MODELS SUMMARY WITH DIAGNOSIS")
print("="*70)

def diagnose_classification(row):
    train_acc = row.get('Train_Acc', np.nan)
    val_acc   = row.get('Val_Acc', np.nan)
    test_acc  = row.get('Accuracy', np.nan)
    train_f1  = row.get('Train_F1', np.nan)
    val_f1    = row.get('Val_F1', np.nan)
    if pd.isna(train_acc) or pd.isna(val_acc):
        return '❓ SIN VALIDACIÓN'
    gap_acc = train_acc - val_acc
    gap_f1  = train_f1 - val_f1
    if train_acc < 0.60 and val_acc < 0.60:
        return '🔴 UNDERFITTING SEVERO'
    if train_acc < 0.75 and val_acc < 0.75:
        return '🔴 UNDERFITTING'
    if train_acc > 0.98 and gap_acc > 0.25:
        return '🚨 OVERFITTING SEVERO'
    if gap_acc > 0.15 or gap_f1 > 0.12:
        return '⚠️ OVERFITTING'
    if gap_acc > 0.08 or gap_f1 > 0.06:
        return '🟡 OVERFITTING LEVE'
    if test_acc < 0.60:
        return '⚠️ BAJO RENDIMIENTO EN TEST'
    return '✅ BUEN AJUSTE'

# Ordenar por Recall_High
df_summary = pd.DataFrame(results).T.sort_values('Recall_High', ascending=False)
df_summary['Diagnosis'] = df_summary.apply(diagnose_classification, axis=1)

print("\n📊 Ordenado por Recall_High (detección de High):")
cols_show = ['F1_weighted', 'F1_macro', 'F1_High', 'Recall_High', 'F1_Empty', 'CV_F1_mean', 'Diagnosis']
print(df_summary[cols_show].to_string())

# Top 3
print(f"\n{'='*70}")
print("🏆 TOP 3 MODELOS (por Recall_High):")
for i, (model_name, row) in enumerate(df_summary.head(3).iterrows()):
    print(f"   {i+1}. {model_name}: Recall_High={row['Recall_High']*100:.1f}% | F1_High={row['F1_High']*100:.1f}%")

best_model_name = df_summary.index[0]
best_metrics    = results[best_model_name]
y_pred_best     = predictions[best_model_name]

print(f"\n   🥇 BEST MODEL: {best_model_name}")
print(f"   Test Accuracy:  {best_metrics['Accuracy']*100:.1f}%")
print(f"   F1_weighted:    {best_metrics['F1_weighted']*100:.1f}%")
print(f"   F1_macro:       {best_metrics['F1_macro']*100:.1f}%")
print(f"   🎯 F1_High:     {best_metrics['F1_High']*100:.1f}%")
print(f"   🎯 Recall_High: {best_metrics['Recall_High']*100:.1f}%")
print(f"   Diagnosis:      {df_summary.loc[best_model_name, 'Diagnosis']}")

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_best)
print(f"\n📊 Confusion Matrix ({best_model_name}):")
df_cm = pd.DataFrame(
    cm,
    index=[f'True {c}' for c in class_names],
    columns=[f'Pred {c}' for c in class_names]
)
print(df_cm.to_string())

# Métricas por clase
print(f"\n📊 Métricas por clase:")
for i, cls_name in enumerate(class_names):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    rec_cls = tp / (tp + fn) if (tp + fn) > 0 else 0
    prec_cls = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_cls = 2 * (prec_cls * rec_cls) / (prec_cls + rec_cls) if (prec_cls + rec_cls) > 0 else 0
    print(f"   {cls_name:<18} Recall={rec_cls:.3f} | Prec={prec_cls:.3f} | F1={f1_cls:.3f} | Support={tp+fn}")

print(f"\n📊 Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=class_names, zero_division=0))

# ==========================================
# 6. GUARDAR RESULTADOS
# ==========================================
print("\n6. SAVING RESULTS...")

with open(os.path.join(OUTPUT_DIR, 'best_model.pkl'), 'wb') as f:
    pickle.dump(trained_models[best_model_name], f)
with open(os.path.join(OUTPUT_DIR, 'all_models.pkl'), 'wb') as f:
    pickle.dump(trained_models, f)

df_preds = pd.DataFrame({
    'Actual': y_test,
    'Actual_label': [class_names[i] for i in y_test],
    'Predicted': y_pred_best,
    'Predicted_label': [class_names[i] for i in y_pred_best],
    'Correct': (y_test == y_pred_best).astype(int),
})
df_preds.to_excel(os.path.join(OUTPUT_DIR, 'predictions.xlsx'), index=False)

df_summary.to_excel(os.path.join(OUTPUT_DIR, 'models_summary.xlsx'))

df_all = pd.DataFrame({'Actual': y_test, 'Actual_label': [class_names[i] for i in y_test]})
for name, y_pred in predictions.items():
    df_all[name] = y_pred
df_all.to_excel(os.path.join(OUTPUT_DIR, 'all_predictions.xlsx'), index=False)

with open(os.path.join(OUTPUT_DIR, 'results.pkl'), 'wb') as f:
    pickle.dump(results, f)
with open(os.path.join(OUTPUT_DIR, 'predictions.pkl'), 'wb') as f:
    pickle.dump(predictions, f)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'), y_test)
np.save(os.path.join(OUTPUT_DIR, 'confusion_matrix.npy'), cm)

# Classification report
report_dict = classification_report(
    y_test, y_pred_best, target_names=class_names,
    output_dict=True, zero_division=0
)
pd.DataFrame(report_dict).transpose().to_excel(
    os.path.join(OUTPUT_DIR, 'classification_report.xlsx')
)

# Diagnosis
df_diagnosis = df_summary[['Train_Acc', 'Val_Acc', 'Acc_Gap', 'Train_F1', 'Val_F1', 'F1_Gap', 'Diagnosis']]
df_diagnosis.to_excel(os.path.join(OUTPUT_DIR, 'diagnosis_summary.xlsx'))

# Feature importance (Random Forest y Extra Trees)
for model_name in ['Random Forest', 'Extra Trees']:
    model = trained_models.get(model_name)
    if model and hasattr(model, 'feature_importances_'):
        try:
            with open(os.path.join(INPUT_DIR, 'selected_features.pkl'), 'rb') as f:
                selected_features = pickle.load(f)
            if len(selected_features) == X_train_raw.shape[1]:
                importances = model.feature_importances_
                df_imp = pd.DataFrame({
                    'Feature': selected_features,
                    'Importance': importances
                }).sort_values('Importance', ascending=False)
                df_imp.to_excel(
                    os.path.join(OUTPUT_DIR, f'feature_importance_{model_name.replace(" ", "_")}.xlsx'), 
                    index=False
                )
                print(f"\n📊 Top 10 Features ({model_name}):")
                print(df_imp.head(10).to_string(index=False))
        except:
            pass

# Config
config = {
    'classroom': CLASSROOM,
    'task': 'classification_4clases',
    'n_classes': 4,
    'class_names': class_names,
    'thresholds': {'Empty': 0, 'Low': '1-20', 'Medium': '21-35', 'High': '>35'},
    'best_model': best_model_name,
    'best_recall_high': best_metrics['Recall_High'],
    'best_f1_high': best_metrics['F1_High'],
    'cv_method': f'StratifiedKFold({n_splits})',
    'n_models_tested': len(models),
}
with open(os.path.join(OUTPUT_DIR, 'config.pkl'), 'wb') as f:
    pickle.dump(config, f)

print(f"\n   ✅ Archivos guardados en '{OUTPUT_DIR}/'")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
    print(f"      {fname} ({size:.0f} KB)")

print("\n" + "="*70)
print("✅ ENTRENAMIENTO Y RESULTADOS COMPLETADOS (sin gráficos)")
print(f"   🏆 Mejor modelo: {best_model_name}")
print(f"   🎯 Recall High: {best_metrics['Recall_High']*100:.1f}%")
print(f"   📊 Modelos evaluados: {len(results)}")
print("="*70)


AULA 1.5 - CLASIFICACIÓN (ENTRENAMIENTO) - 4 CLASES

1. LOADING DATA...
   Train: 125, Test: 55

2. CONVERTING TO CATEGORIES...
   Distribución de clases:
      Empty(0): train=39 (31.2%), test=17 (30.9%)
      Low(1-20): train=33 (26.4%), test=19 (34.5%)
      Medium(21-35): train=30 (24.0%), test=12 (21.8%)
      High(36+): train=23 (18.4%), test=7 (12.7%)

3. DEFINING MODELS & VALIDATION...
   Modelos: 11 | CV: StratifiedKFold(5)
   Con normalización: ['KNN (k=10)', 'KNN (k=5)', 'Logistic Regression', 'MLP (Neural Network)', 'Naive Bayes', 'SVM']
   Sin normalización: ['AdaBoost', 'Decision Tree', 'Extra Trees', 'Gradient Boosting', 'Random Forest']

4. TRAINING MODELS WITH DIAGNOSIS...

   ▶ Random Forest (raw)...
      Test Acc: 0.6545 | F1w: 0.6626 | F1m: 0.6101
      Empty: F1=0.9697 | Low: F1=0.6667 | Medium: F1=0.3333 | High: F1=0.4706
      Recall_High: 0.5714 | Recall_Empty: 0.9412
      CV F1: 0.7289 ± 0.0632
      Train Acc: 1.0000 | Val Acc: 0.7280 | Gap: 0.2720

   ▶ Ex